[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LP-D/claude/blob/main/notebooks/research/VIX_VAR_MACRO.ipynb)


# VIX Macro VAR v1 — liens dynamiques VIX ↔ macro (structurel, hors pipeline ML)

**Rôle.** Nouveau notebook indépendant, à lancer après `VIX_FINAL_FEATURES` (dont il réutilise
l'univers de tickers pour la partie VIX). Contrairement à tout le reste du projet — qui prédit
une classe (direction/amplitude) via ML/DL — celui-ci fait de l'**analyse structurelle** :
comprendre *comment et pourquoi* le VIX est lié à un petit ensemble de variables macro, via un
modèle VAR (Vecteur Autorégressif).

**Variables** : VIX (`^VIX`), pente de la courbe des taux (`T10Y2Y`), inflation anticipée à 10 ans
(`T10YIE`, breakeven), conditions financières (`NFCI`), taux effectif des fonds fédéraux (`EFFR`)
— toutes issues de FRED sauf le VIX (Yahoo Finance).

**Méthodologie** (inspirée du cours d'Advanced Time Series Econometrics — VAR, causalité de
Granger, IRF, FEVD, cointégration — mais implémentée ici indépendamment en Python avec
`statsmodels`, pour ce projet de recherche, et non comme reproduction d'un devoir) :
1. Test de stationnarité (ADF) sur chaque série.
2. Test de cointégration de Johansen — si les séries I(1) sont cointégrées, un VAR en différences
   (utilisé ensuite dans ce notebook) reste valide mais laisse de côté la relation de long terme ;
   ce cas est signalé explicitement comme piste pour un futur notebook VECM dédié.
3. Sélection de l'ordre de retard p (AIC/BIC/HQIC), estimation du VAR, vérification de la
   stabilité (valeurs propres de la matrice compagnon).
4. Tests de causalité de Granger (toutes les paires).
5. Fonctions de réponse impulsionnelle (IRF) orthogonalisées (décomposition de Cholesky) et
   décomposition de la variance de l'erreur de prévision (FEVD).

**Ce que ce notebook ne fait PAS** : il ne produit pas de nouvelle feature validée en walk-forward
pour le pipeline ML — il identifie des pistes (variables macro qui causent le VIX au sens de
Granger, avec quel délai) à tester plus tard avec la méthodologie SHAP + walk-forward déjà en
place dans ce projet.


In [1]:
import subprocess, sys
pkgs = ['yfinance', 'pandas_datareader', 'statsmodels', 'xlsxwriter', 'matplotlib']
subprocess.run([sys.executable,'-m','pip','install','-q']+pkgs, check=False)
print("Installation OK")


Installation OK


In [2]:
import os, json, time, warnings, random, subprocess
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import yfinance as yf
import pandas_datareader.data as web
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.vector_ar.var_model import VAR
from statsmodels.tsa.vector_ar.vecm import coint_johansen

SEED = 42; random.seed(SEED); np.random.seed(SEED)

NOTEBOOK_NAME = 'VIX_MACRO_VAR'
NOTEBOOK_VERSION = 'v1'

CONFIG = {
    'vix_ticker': '^VIX',
    'fred_series': {'T10Y2Y': 'T10Y2Y', 'T10YIE': 'T10YIE', 'NFCI': 'NFCI', 'EFFR': 'EFFR'},
    'start_date': '2003-01-01',
    'adf_alpha': 0.05,
    'maxlags': 10,
    'irf_periods': 20,
    'fevd_periods': 20,
    'forecast_steps': 60,
    'johansen_det_order': 0,   # constant term, no trend
}
GITHUB_REPO = 'LP-D/claude'
RESULTS_BRANCH = 'results/vix-macro-var'

print(f"{NOTEBOOK_NAME} {NOTEBOOK_VERSION} | variables: VIX + "
      f"{list(CONFIG['fred_series'].keys())} | depuis {CONFIG['start_date']}")


VIX_MACRO_VAR v1 | variables: VIX + ['T10Y2Y', 'T10YIE', 'NFCI', 'EFFR'] | depuis 2003-01-01


In [3]:
# ============================================================
# CHARGEMENT DES DONNÉES : VIX (Yahoo Finance) + macro (FRED)
# ============================================================
def load_data():
    t0 = time.time()
    vix = yf.download(CONFIG['vix_ticker'], start=CONFIG['start_date'], auto_adjust=True, progress=False)['Close']
    if isinstance(vix, pd.DataFrame):
        vix = vix.iloc[:, 0]
    vix.name = 'VIX'
    df = vix.to_frame()
    print(f"  VIX: {len(df)} obs ({time.time()-t0:.1f}s)")

    for name, sid in CONFIG['fred_series'].items():
        try:
            s = web.DataReader(sid, 'fred', CONFIG['start_date']).squeeze()
            df[name] = s.reindex(df.index, method='ffill')
        except Exception as e:
            print(f"  [WARN] FRED {sid}: {str(e)[:100]}")

    expected = ['VIX'] + list(CONFIG['fred_series'].keys())
    missing = [c for c in expected if c not in df.columns]
    if missing:
        print(f"  [WARN] séries manquantes: {missing}")
    df = df.ffill().dropna()
    print(f"  Total aligné: {df.shape} ({time.time()-t0:.1f}s) | colonnes: {list(df.columns)}")
    return df

df_raw = load_data()
VAR_COLS = list(df_raw.columns)
print(df_raw.tail())


  VIX: 5928 obs (0.7s)
  Total aligné: (5927, 5) (1.0s) | colonnes: ['VIX', 'T10Y2Y', 'T10YIE', 'NFCI', 'EFFR']
                  VIX  T10Y2Y  T10YIE   NFCI  EFFR
Date                                              
2026-07-20  18.650000    0.39    2.25 -0.552  3.63
2026-07-21  17.049999    0.37    2.26 -0.552  3.63
2026-07-22  16.639999    0.36    2.28 -0.552  3.63
2026-07-23  18.700001    0.34    2.28 -0.552  3.63
2026-07-24  18.830000    0.34    2.28 -0.552  3.63


In [4]:
# ============================================================
# STATIONNARITÉ (ADF) — détermine l'ordre d'intégration de chaque série
# ============================================================
def adf_report(s, name):
    s = s.dropna()
    stat, pval, *_ = adfuller(s, autolag='AIC')
    stationary = pval < CONFIG['adf_alpha']
    print(f"  {name:10s} ADF stat={stat:8.3f}  p-value={pval:.4f}  "
          f"{'I(0) stationnaire' if stationary else 'I(1) probable (racine unitaire)'}")
    return stationary

print("### ADF en niveau ###")
levels_stationary = {c: adf_report(df_raw[c], c) for c in VAR_COLS}

df_diff = df_raw.diff().dropna()
print("\n### ADF en différence première (au cas où) ###")
diff_stationary = {c: adf_report(df_diff[c], c) for c in VAR_COLS}

I1_COLS = [c for c, ok in levels_stationary.items() if not ok]
I0_COLS = [c for c, ok in levels_stationary.items() if ok]
print(f"\nI(0) en niveau: {I0_COLS or 'aucune'}")
print(f"I(1) probable (à tester en cointégration avant de différencier): {I1_COLS or 'aucune'}")


### ADF en niveau ###
  VIX        ADF stat=  -6.076  p-value=0.0000  I(0) stationnaire
  T10Y2Y     ADF stat=  -1.744  p-value=0.4087  I(1) probable (racine unitaire)
  T10YIE     ADF stat=  -3.821  p-value=0.0027  I(0) stationnaire
  NFCI       ADF stat=  -3.941  p-value=0.0018  I(0) stationnaire
  EFFR       ADF stat=  -1.286  p-value=0.6357  I(1) probable (racine unitaire)

### ADF en différence première (au cas où) ###
  VIX        ADF stat= -15.862  p-value=0.0000  I(0) stationnaire
  T10Y2Y     ADF stat= -12.905  p-value=0.0000  I(0) stationnaire
  T10YIE     ADF stat= -12.962  p-value=0.0000  I(0) stationnaire
  NFCI       ADF stat=  -7.009  p-value=0.0000  I(0) stationnaire
  EFFR       ADF stat=  -8.700  p-value=0.0000  I(0) stationnaire

I(0) en niveau: ['VIX', 'T10YIE', 'NFCI']
I(1) probable (à tester en cointégration avant de différencier): ['T10Y2Y', 'EFFR']


In [5]:
# ============================================================
# TEST DE COINTÉGRATION DE JOHANSEN (trace test, seuil 95%)
# rang(Π) : si 0 -> VAR en différences légitime ; si 0<r<K -> relation(s) de
# long terme laissée(s) de côté par le VAR en différences (piste VECM future) ;
# si r=K -> les séries seraient stationnaires en niveau.
# ============================================================
johansen = coint_johansen(df_raw[VAR_COLS].values, CONFIG['johansen_det_order'], CONFIG['maxlags'])
trace_stat = johansen.lr1
trace_crit = johansen.cvt[:, 1]  # colonne 95%
n_vars = len(VAR_COLS)

print("### Test de la trace (Johansen), seuil 95% ###")
coint_rank = 0
for i in range(n_vars):
    reject = trace_stat[i] > trace_crit[i]
    print(f"  r<={i}: trace={trace_stat[i]:8.3f}  crit_95%={trace_crit[i]:8.3f}  "
          f"{'rejette H0 (r>' + str(i) + ')' if reject else 'ne rejette pas H0'}")
    if reject:
        coint_rank = i + 1

print(f"\n[JOHANSEN] Rang de cointégration estimé: r={coint_rank} (sur {n_vars} variables)")
if coint_rank == 0:
    print("  -> Aucune relation de cointégration détectée : VAR en différences légitime, "
          "pas de relation de long terme à corriger.")
elif coint_rank == n_vars:
    print("  -> Rang plein : les séries seraient stationnaires en niveau (cas rare compte "
          "tenu du test ADF ci-dessus).")
else:
    print(f"  -> {coint_rank} relation(s) de cointégration détectée(s). Le VAR en différences "
          "utilisé ci-dessous reste valide pour l'analyse court terme (Granger, IRF, FEVD) "
          "mais laisse de côté la relation de long terme (Π = α·β') — piste explicite pour "
          "un futur notebook VECM dédié, pas traitée ici.")
COINT_RANK = coint_rank


### Test de la trace (Johansen), seuil 95% ###
  r<=0: trace= 231.345  crit_95%=  69.819  rejette H0 (r>0)
  r<=1: trace=  94.861  crit_95%=  47.855  rejette H0 (r>1)
  r<=2: trace=  30.992  crit_95%=  29.796  rejette H0 (r>2)
  r<=3: trace=  15.493  crit_95%=  15.494  ne rejette pas H0
  r<=4: trace=   5.591  crit_95%=   3.841  rejette H0 (r>4)

[JOHANSEN] Rang de cointégration estimé: r=5 (sur 5 variables)
  -> Rang plein : les séries seraient stationnaires en niveau (cas rare compte tenu du test ADF ci-dessus).


In [6]:
# ============================================================
# TRANSFORMATION ET SÉLECTION DE L'ORDRE DE RETARD p (AIC/BIC/HQIC)
# ============================================================
if I1_COLS:
    df_var = df_raw.diff().dropna()
    VAR_TRANSFORM = 'différences premières (au moins une série I(1) détectée)'
else:
    df_var = df_raw.copy()
    VAR_TRANSFORM = 'niveaux (toutes les séries I(0))'
print(f"[VAR] Données utilisées: {VAR_TRANSFORM} — {df_var.shape}")

model_sel = VAR(df_var[VAR_COLS])
sel = model_sel.select_order(CONFIG['maxlags'])
print(sel.summary())

lag_aic, lag_bic, lag_hqic = sel.aic, sel.bic, sel.hqic
print(f"\nOrdre retenu — AIC={lag_aic}  BIC={lag_bic}  HQIC={lag_hqic}")
VAR_LAG = lag_aic if lag_aic and lag_aic > 0 else max(lag_bic or 1, 1)
print(f"[SÉLECTION] p retenu (AIC, minimum 1): p={VAR_LAG}")


[VAR] Données utilisées: différences premières (au moins une série I(1) détectée) — (5926, 5)


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


 VAR Order Selection (* highlights the minimums)  
       AIC         BIC         FPE         HQIC   
--------------------------------------------------
0       -26.02      -26.01   5.028e-12      -26.01
1       -26.10      -26.07   4.621e-12      -26.09
2       -26.15      -26.08   4.411e-12      -26.13
3       -26.19      -26.10   4.215e-12      -26.16
4       -26.24      -26.12   4.036e-12      -26.19
5       -27.48      -27.34   1.158e-12      -27.43
6       -27.60      -27.43   1.029e-12      -27.54
7       -27.64      -27.44   9.915e-13      -27.57
8       -27.65      -27.42   9.790e-13      -27.57
9       -27.67      -27.41   9.653e-13      -27.58
10     -27.74*     -27.45*  8.970e-13*     -27.64*
--------------------------------------------------

Ordre retenu — AIC=10  BIC=10  HQIC=10
[SÉLECTION] p retenu (AIC, minimum 1): p=10


In [7]:
# ============================================================
# ESTIMATION DU VAR(p) ET VÉRIFICATION DE LA STABILITÉ
# ============================================================
var_model = VAR(df_var[VAR_COLS])
var_res = var_model.fit(VAR_LAG)
print(var_res.summary())

stable = var_res.is_stable()
roots = var_res.roots
print(f"\n[STABILITÉ] VAR({VAR_LAG}) {'STABLE' if stable else 'INSTABLE'} — racines du polynôme "
      f"caractéristique inverse (doivent être strictement hors du cercle unité, |racine|>1)")
print(f"  min |racine| = {np.min(np.abs(roots)):.4f}")
if not stable:
    print("  [WARN] VAR instable : IRF/FEVD ci-dessous restent calculables mais leur "
          "interprétation en tant que réponses convergentes n'est plus garantie.")


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Fri, 24, Jul, 2026
Time:                     07:37:48
--------------------------------------------------------------------
No. of Equations:         5.00000    BIC:                   -27.4516
Nobs:                     5916.00    HQIC:                  -27.6396
Log likelihood:           40337.0    FPE:                8.96983e-13
AIC:                     -27.7397    Det(Omega_mle):     8.59300e-13
--------------------------------------------------------------------
Results for equation VIX
                coefficient       std. error           t-stat            prob
-----------------------------------------------------------------------------
const             -0.000908         0.023009           -0.039           0.969
L1.VIX            -0.202220         0.013594          -14.876           0.000
L1.T10Y2Y          0.404646         0.619607            0.653           0.

### Rappels théoriques (cours Advanced Time Series Econometrics, Dr. Duc Thi LUU)

**VAR(p)** : `yt = c + A1·yt-1 + ... + Ap·yt-p + ut`. Stabilité ⟺ les racines du polynôme
caractéristique inverse `det(I_K - A1·z - ... - Ap·z^p)` sont strictement hors du cercle
unité (équivalent : valeurs propres de la matrice compagnon strictement dans le cercle
unité). Représentation VMA(∞) : `yt = μ + Σ Φi·ut-i`.

**Causalité de Granger** : `bt` ne cause pas `at` au sens de Granger ssi les blocs
hors-diagonale correspondants du VAR sont nuls à tous les retards — testé ici via
`VARResults.test_causality(caused, causing, kind='f')` (équivalent direct du `causality()`
du package R `vars` utilisé en cours).

**IRF orthogonalisées** : les chocs `ut` étant typiquement corrélés (`Σu` non diagonale),
on décompose `Σu = P·P'` (Cholesky) et on définit `wt = P⁻¹·ut` (non corrélés, variance
unitaire), donnant `Θi = Φi·P`. L'ordre des variables dans la décomposition de Cholesky
influence le résultat (le VIX est placé en dernier ci-dessous : on isole l'effet d'un choc
macro *avant* toute réaction contemporaine du VIX).

**FEVD** : la variance totale de l'erreur de prévision de `y_j,t+h` se décompose comme
`Σk Σ(i=0 à h-1) Θjk,i²` ; la part relative de la source `k` est ce terme divisé par la
variance totale — c'est la mesure utilisée ci-dessous pour quantifier la contribution de
chaque variable macro à l'incertitude de prévision du VIX.


In [8]:
# ============================================================
# CAUSALITÉ DE GRANGER (toutes les paires ordonnées, test F)
# ============================================================
granger_rows = []
for caused in VAR_COLS:
    for causing in VAR_COLS:
        if caused == causing:
            continue
        try:
            test = var_res.test_causality(caused, [causing], kind='f')
            granger_rows.append({
                'causing': causing, 'caused': caused,
                'F_stat': round(float(test.test_statistic), 4),
                'p_value': round(float(test.pvalue), 4),
                'significant_5pct': bool(test.pvalue < 0.05),
            })
        except Exception as e:
            print(f"  [WARN] {causing} -> {caused}: {str(e)[:100]}")

df_granger = pd.DataFrame(granger_rows).sort_values('p_value')
print(df_granger.to_string(index=False))

sig_to_vix = df_granger[(df_granger['caused'] == 'VIX') & (df_granger['significant_5pct'])]
print(f"\n[GRANGER -> VIX] Variables qui causent le VIX au sens de Granger (p<5%): "
      f"{sig_to_vix['causing'].tolist() or 'aucune'}")

sig_from_vix = df_granger[(df_granger['causing'] == 'VIX') & (df_granger['significant_5pct'])]
print(f"[GRANGER VIX ->] Variables causées par le VIX au sens de Granger (p<5%): "
      f"{sig_from_vix['caused'].tolist() or 'aucune'}")


causing caused  F_stat  p_value  significant_5pct
   EFFR    VIX  8.1107   0.0000              True
   NFCI    VIX 17.7861   0.0000              True
   NFCI T10Y2Y  4.5774   0.0000              True
    VIX T10YIE  3.9914   0.0000              True
    VIX   NFCI  6.8876   0.0000              True
 T10Y2Y   NFCI  3.7699   0.0000              True
   NFCI T10YIE 13.0248   0.0000              True
 T10Y2Y   EFFR  4.3933   0.0000              True
   NFCI   EFFR 11.9572   0.0000              True
 T10YIE   NFCI  3.4051   0.0002              True
    VIX   EFFR  3.2242   0.0004              True
    VIX T10Y2Y  2.9578   0.0010              True
   EFFR T10Y2Y  2.9473   0.0010              True
   EFFR T10YIE  2.7765   0.0020              True
 T10Y2Y T10YIE  1.9751   0.0317              True
 T10YIE    VIX  1.8051   0.0542             False
 T10YIE   EFFR  1.4668   0.1447             False
 T10Y2Y    VIX  1.4425   0.1545             False
   EFFR   NFCI  1.3163   0.2148             False


In [9]:
# ============================================================
# IRF ORTHOGONALISÉES (Cholesky) ET FEVD — focus sur la réponse du VIX
# ============================================================
irf = var_res.irf(CONFIG['irf_periods'])
vix_i = VAR_COLS.index('VIX')

try:
    fig = irf.plot(orth=True, response='VIX', figsize=(10, 8))
    plt.tight_layout()
    plt.savefig('irf_vix_response.png', dpi=100)
    plt.close()
    print("[SAVE] irf_vix_response.png — réponse du VIX à un choc orthogonalisé sur chaque variable")
except Exception as e:
    print(f"[WARN plot IRF] {e}")

theta = irf.orth_irfs  # Θ_i = Φ_i · P, shape (periods+1, n_vars, n_vars)
cum_effect = {src: float(np.sum(theta[:, vix_i, j])) for j, src in enumerate(VAR_COLS)}
print("\n[IRF] Effet cumulé (sur irf_periods) d'un choc orthogonalisé unitaire de chaque "
      "variable sur le VIX:")
for src, val in sorted(cum_effect.items(), key=lambda x: -abs(x[1])):
    print(f"  choc sur {src:10s} -> effet cumulé sur VIX = {val:+.4f}")

fevd = var_res.fevd(CONFIG['fevd_periods'])
fevd_vix = pd.DataFrame(fevd.decomp[vix_i], columns=VAR_COLS)
fevd_vix.index = range(1, len(fevd_vix) + 1)
print(f"\n[FEVD] Décomposition de la variance de l'erreur de prévision du VIX à h="
      f"{CONFIG['fevd_periods']}:")
last_h = fevd_vix.iloc[-1].sort_values(ascending=False)
for src, pct in last_h.items():
    print(f"  {src:10s}: {pct:.1%} de la variance de l'erreur de prévision du VIX")


[SAVE] irf_vix_response.png — réponse du VIX à un choc orthogonalisé sur chaque variable

[IRF] Effet cumulé (sur irf_periods) d'un choc orthogonalisé unitaire de chaque variable sur le VIX:
  choc sur VIX        -> effet cumulé sur VIX = +1.2122
  choc sur NFCI       -> effet cumulé sur VIX = +0.6224
  choc sur T10Y2Y     -> effet cumulé sur VIX = +0.1128
  choc sur T10YIE     -> effet cumulé sur VIX = +0.0811
  choc sur EFFR       -> effet cumulé sur VIX = -0.0430

[FEVD] Décomposition de la variance de l'erreur de prévision du VIX à h=20:
  VIX       : 96.3% de la variance de l'erreur de prévision du VIX
  NFCI      : 1.8% de la variance de l'erreur de prévision du VIX
  EFFR      : 1.4% de la variance de l'erreur de prévision du VIX
  T10Y2Y    : 0.3% de la variance de l'erreur de prévision du VIX
  T10YIE    : 0.3% de la variance de l'erreur de prévision du VIX


In [10]:
# ============================================================
# PISTES POUR LE PIPELINE ML — NON VALIDÉES, à tester via SHAP + walk-forward
# ============================================================
print("Ce notebook identifie des candidats macro pour le pipeline ML (VIX_FINAL_FEATURES /")
print("VIX_FINAL_ML_SCAN) sur la base de la causalité de Granger et de l'IRF ci-dessus.")
print("Ces candidats ne sont PAS validés par la méthodologie walk-forward + SHAP du projet —")
print("ce sont des pistes de recherche, à confirmer ou infirmer séparément.\n")

bridge_rows = []
for _, row in sig_to_vix.iterrows():
    bridge_rows.append({
        'candidate_feature': f"{row['causing']}_lag{VAR_LAG}",
        'granger_p_value': row['p_value'],
        'irf_cumulative_effect_on_vix': round(cum_effect.get(row['causing'], np.nan), 4),
        'fevd_share_of_vix_variance_pct': round(float(last_h.get(row['causing'], np.nan)) * 100, 2),
        'suggestion': f"lag {VAR_LAG} de {row['causing']} (ou sa variation) comme feature "
                       f"candidate, à ajouter au pool SHAP de VIX_FINAL_FEATURES et "
                       f"re-scanner en walk-forward",
    })
df_bridge = pd.DataFrame(bridge_rows)
if len(df_bridge):
    print(df_bridge.to_string(index=False))
else:
    print("[BRIDGE] Aucune variable ne cause le VIX au sens de Granger à 5% — pas de piste "
          "nouvelle à proposer pour le pipeline ML à partir de cette analyse.")


Ce notebook identifie des candidats macro pour le pipeline ML (VIX_FINAL_FEATURES /
VIX_FINAL_ML_SCAN) sur la base de la causalité de Granger et de l'IRF ci-dessus.
Ces candidats ne sont PAS validés par la méthodologie walk-forward + SHAP du projet —
ce sont des pistes de recherche, à confirmer ou infirmer séparément.

candidate_feature  granger_p_value  irf_cumulative_effect_on_vix  fevd_share_of_vix_variance_pct                                                                                                                           suggestion
       EFFR_lag10              0.0                       -0.0430                            1.42 lag 10 de EFFR (ou sa variation) comme feature candidate, à ajouter au pool SHAP de VIX_FINAL_FEATURES et re-scanner en walk-forward
       NFCI_lag10              0.0                        0.6224                            1.82 lag 10 de NFCI (ou sa variation) comme feature candidate, à ajouter au pool SHAP de VIX_FINAL_FEATURES et re-scanner en wa

In [11]:
# ============================================================
# EXPORT (xlsx) ET PUSH SUR LA BRANCHE results/vix-macro-var
# ============================================================
try:
    with pd.ExcelWriter('VIX_VAR_MACRO_report.xlsx', engine='xlsxwriter') as w:
        df_raw[VAR_COLS].to_excel(w, 'Data_niveaux')
        df_granger.to_excel(w, 'Granger', index=False)
        pd.DataFrame([{'variable': k, 'effet_cumule_irf_sur_VIX': v}
                      for k, v in cum_effect.items()]).to_excel(w, 'IRF_cumule', index=False)
        fevd_vix.to_excel(w, 'FEVD_VIX')
        (df_bridge if len(df_bridge) else pd.DataFrame()).to_excel(w, 'Pistes_ML', index=False)
        pd.DataFrame([{'rang_cointegration_johansen': COINT_RANK, 'var_lag_retenu': VAR_LAG,
                        'var_stable': stable, 'transformation': VAR_TRANSFORM}]
                     ).to_excel(w, 'Meta', index=False)
    print("[SAVE] VIX_VAR_MACRO_report.xlsx")
except Exception as e:
    print(f"[WARN Export] {e}")

GITHUB_TOKEN = None
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN')

_PUSH_WORKDIR = "/content/_vix_var_macro_push"

def push_report():
    if not GITHUB_TOKEN or not os.path.exists('VIX_VAR_MACRO_report.xlsx'):
        print("[SKIP] Pas de token ou pas de rapport à pousser.")
        return
    try:
        subprocess.run(["rm", "-rf", _PUSH_WORKDIR], check=False)
        url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
        clone = subprocess.run(["git", "clone", url, _PUSH_WORKDIR], capture_output=True, text=True)
        if clone.returncode != 0:
            print(f"[WARN] clone: {clone.stderr[-300:]}"); return
        exists = subprocess.run(["git", "-C", _PUSH_WORKDIR, "ls-remote", "--exit-code", "--heads",
                                  "origin", RESULTS_BRANCH], capture_output=True, text=True)
        if exists.returncode == 0:
            subprocess.run(["git", "-C", _PUSH_WORKDIR, "checkout", "-B", RESULTS_BRANCH,
                             f"origin/{RESULTS_BRANCH}"], check=True)
        else:
            subprocess.run(["git", "-C", _PUSH_WORKDIR, "checkout", "-B", RESULTS_BRANCH], check=True)
        for f in ['VIX_VAR_MACRO_report.xlsx', 'irf_vix_response.png']:
            if os.path.exists(f):
                subprocess.run(["cp", f, f"{_PUSH_WORKDIR}/{f}"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "config", "user.email",
                         "vix-colab@users.noreply.github.com"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "config", "user.name",
                         "VIX Macro VAR Colab run"], check=True)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "add", "VIX_VAR_MACRO_report.xlsx",
                         "irf_vix_response.png"], check=False)
        subprocess.run(["git", "-C", _PUSH_WORKDIR, "commit", "-m",
                       f"Rapport VAR Macro — {pd.Timestamp.now():%Y-%m-%d %H:%M}"],
                       capture_output=True, text=True)
        push = subprocess.run(["git", "-C", _PUSH_WORKDIR, "push", "origin", RESULTS_BRANCH],
                              capture_output=True, text=True)
        if push.returncode == 0:
            print(f"[PUSH OK] VIX_VAR_MACRO_report.xlsx sur '{RESULTS_BRANCH}'")
        else:
            print(f"[WARN] {push.stderr[-300:]}")
    except Exception as e:
        print(f"[WARN] {e}")

push_report()


[SAVE] VIX_VAR_MACRO_report.xlsx
[PUSH OK] VIX_VAR_MACRO_report.xlsx sur 'results/vix-macro-var'
